# 🧬 Kindred Careers — Embedding Model Fine-Tuning Demonstration

**Project**: Kindred Careers — A Swipe-Based Job Finder Application  
**Purpose**: Fine-tune the `all-MiniLM-L6-v2` sentence-transformer model on Egyptian tech/creative job-market domain data to improve semantic matching between candidate profiles and job listings.

---

### Why Fine-Tune?

The base `all-MiniLM-L6-v2` model is a general-purpose sentence embedding model trained on broad internet text. While it understands English semantics well, it has **no specialized understanding** of:

- Egyptian tech company names (Instabug, Swvl, Breadfast, Fawry)
- Regional job market terminology and skill associations
- The specific text formatting our application uses for job/profile embeddings

By fine-tuning on carefully curated **profile ↔ job listing** pairs from our target domain, we teach the model to produce **tighter clusters** for matching profiles and jobs, directly improving the quality of our swipe-based recommendations.

### Key Properties

| Property | Value |
|----------|-------|
| Base Model | `all-MiniLM-L6-v2` |
| Output Dimensions | 384 (preserved after fine-tuning) |
| Model Size | ~90 MB |
| Training Approach | Contrastive learning with CosineSimilarityLoss |
| Training Data | 20 domain-specific pairs (16 positive, 4 negative) |
| Execution | 100% local — no external API calls |

## Step 1: Load the Base Pre-Trained Model

We start by loading the original `all-MiniLM-L6-v2` model. This is the **starting point** before any fine-tuning. We verify it produces 384-dimensional embeddings.

In [ ]:
import time
import numpy as np
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

BASE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print(f"\U0001f4e6 Loading base model: {BASE_MODEL_NAME}...")
start = time.time()
base_model = SentenceTransformer(BASE_MODEL_NAME)
print(f"\u2705 Loaded in {time.time() - start:.1f}s")
print(f"   Embedding dimensions: {base_model.get_sentence_embedding_dimension()}")
print(f"   Max sequence length: {base_model.max_seq_length}")

## Step 2: Define Domain-Specific Training Data

Each training example is a pair of texts with a **similarity label** (0.0 to 1.0):

- **Positive pairs** (label 0.85–0.95): A candidate profile that **matches** a job listing
- **Negative pairs** (label 0.10–0.20): A candidate profile that does **NOT** match a job listing

The text formatting mirrors our production system:
- **Profile format**: `"Professional candidate with expertise in {fields}. Core skills include: {skills}..."`
- **Job format**: `"Job Title: {title}. Company: {company}. Required Skills: {skills}..."`

### Coverage

| Domain | # Positive | Companies |
|--------|-----------|----------|
| Software Engineering | 3 | Instabug, Vodafone Egypt, Breadfast |
| Mobile Development | 2 | Swvl, Halan |
| AI / Machine Learning | 2 | Valeo Egypt, Fawry |
| UI/UX Design | 2 | Breadfast, Paymob |
| Video & Creative | 2 | MO4 Network, Tribal DDB |
| Digital Marketing | 1 | Jumia Egypt |
| DevOps | 1 | Si-Ware Systems |
| Data Analysis | 1 | Orange Egypt |
| Cross-domain | 2 | MaxAB/Capiter, Robusta Studio |

In [ ]:
train_examples = [
    # ---- Positive Pairs: Software Engineering ----
    InputExample(
        texts=[
            "Professional candidate with expertise in Software Engineering. "
            "Core skills include: Python, Django, PostgreSQL, REST APIs. "
            "Work history: Backend Developer at Instabug. "
            "Prefers Remote work in Cairo, Egypt.",
            "Job Title: Senior Python Developer. "
            "Company: Instabug. Industry: Technology. "
            "Required Skills: Python, Django, REST APIs, PostgreSQL. "
            "Work Mode: Remote. Location: Cairo, Egypt. "
            "Description: Build scalable backend services for a leading Egyptian SaaS company.",
        ], label=0.92),
    InputExample(
        texts=[
            "Professional candidate with expertise in Software Engineering. "
            "Core skills include: Java, Spring Boot, Microservices, AWS. "
            "Work history: Software Engineer at Vodafone Egypt. "
            "Prefers Hybrid work in Cairo, Egypt.",
            "Job Title: Java Backend Engineer. "
            "Company: Vodafone Egypt. Industry: Telecommunications. "
            "Required Skills: Java, Spring Boot, Microservices, Docker. "
            "Work Mode: Hybrid. Location: Smart Village, Egypt. "
            "Description: Design high-throughput telecom backend systems.",
        ], label=0.90),
    InputExample(
        texts=[
            "Professional candidate with expertise in Software Engineering. "
            "Core skills include: Node.js, TypeScript, Express, MongoDB. "
            "Work history: Full Stack Developer at Breadfast. "
            "Prefers Onsite work in Cairo, Egypt.",
            "Job Title: Full Stack JavaScript Developer. "
            "Company: Breadfast. Industry: E-Commerce. "
            "Required Skills: Node.js, TypeScript, React, MongoDB. "
            "Work Mode: Onsite. Location: Cairo, Egypt. "
            "Description: Build and scale the grocery delivery platform.",
        ], label=0.91),

    # ---- Positive Pairs: Mobile Development ----
    InputExample(
        texts=[
            "Professional candidate with expertise in Mobile Development. "
            "Core skills include: Flutter, Dart, Firebase, REST APIs. "
            "Work history: Mobile Developer at Swvl. "
            "Prefers Remote work in Cairo, Egypt.",
            "Job Title: Flutter Developer. "
            "Company: Swvl. Industry: Transportation Technology. "
            "Required Skills: Flutter, Dart, Firebase, REST APIs, Git. "
            "Work Mode: Remote. Location: Cairo, Egypt. "
            "Description: Develop cross-platform mobile applications for mass transit.",
        ], label=0.93),
    InputExample(
        texts=[
            "Professional candidate with expertise in Mobile Development. "
            "Core skills include: React Native, JavaScript, Redux, Firebase. "
            "Work history: Mobile Engineer at Halan. "
            "Prefers Hybrid work in Cairo, Egypt.",
            "Job Title: React Native Developer. "
            "Company: Halan. Industry: FinTech. "
            "Required Skills: React Native, JavaScript, Redux, REST APIs. "
            "Work Mode: Hybrid. Location: Cairo, Egypt. "
            "Description: Build mobile fintech solutions for unbanked users.",
        ], label=0.90),

    # ---- Positive Pairs: AI / Machine Learning ----
    InputExample(
        texts=[
            "Professional candidate with expertise in Artificial Intelligence. "
            "Core skills include: Python, TensorFlow, PyTorch, NLP, Computer Vision. "
            "Work history: ML Engineer at Valeo Egypt. "
            "Prefers Hybrid work in Cairo, Egypt.",
            "Job Title: Machine Learning Engineer. "
            "Company: Valeo Egypt. Industry: Automotive Technology. "
            "Required Skills: Python, TensorFlow, PyTorch, Computer Vision. "
            "Work Mode: Hybrid. Location: Cairo, Egypt. "
            "Description: Develop deep learning models for autonomous driving.",
        ], label=0.93),
    InputExample(
        texts=[
            "Professional candidate with expertise in Data Science. "
            "Core skills include: Python, pandas, scikit-learn, SQL, Tableau. "
            "Work history: Data Analyst at Fawry. "
            "Prefers Onsite work in Cairo, Egypt.",
            "Job Title: Data Scientist. "
            "Company: Fawry. Industry: FinTech. "
            "Required Skills: Python, pandas, SQL, Machine Learning, Statistics. "
            "Work Mode: Onsite. Location: Cairo, Egypt. "
            "Description: Analyze transaction data and build predictive models.",
        ], label=0.88),

    # ---- Positive Pairs: UI/UX Design ----
    InputExample(
        texts=[
            "Professional candidate with expertise in UI/UX Design. "
            "Core skills include: Figma, Adobe XD, User Research, Prototyping. "
            "Work history: UX Designer at Breadfast. "
            "Prefers Remote work in Cairo, Egypt.",
            "Job Title: Senior UX Designer. "
            "Company: Breadfast. Industry: E-Commerce. "
            "Required Skills: Figma, Prototyping, User Research, Design Systems. "
            "Work Mode: Remote. Location: Cairo, Egypt. "
            "Description: Lead UX design for a fast-growing grocery delivery app.",
        ], label=0.90),
    InputExample(
        texts=[
            "Professional candidate with expertise in UI/UX Design. "
            "Core skills include: Sketch, InVision, Wireframing, Interaction Design. "
            "Work history: Product Designer at Paymob. "
            "Prefers Hybrid work in Cairo, Egypt.",
            "Job Title: Product Designer. "
            "Company: Paymob. Industry: FinTech. "
            "Required Skills: Figma, Sketch, Design Thinking, Wireframing. "
            "Work Mode: Hybrid. Location: Cairo, Egypt. "
            "Description: Design intuitive payment interfaces for the MENA region.",
        ], label=0.88),

    # ---- Positive Pairs: Video Production & Creative ----
    InputExample(
        texts=[
            "Professional candidate with expertise in Video Production. "
            "Core skills include: Adobe Premiere Pro, After Effects, DaVinci Resolve. "
            "Work history: Video Editor at MO4 Network. "
            "Prefers Onsite work in Cairo, Egypt.",
            "Job Title: Senior Video Editor. "
            "Company: MO4 Network. Industry: Digital Media. "
            "Required Skills: Premiere Pro, After Effects, Motion Graphics. "
            "Work Mode: Onsite. Location: Cairo, Egypt. "
            "Description: Produce high-quality digital content for Arabic YouTube.",
        ], label=0.91),
    InputExample(
        texts=[
            "Professional candidate with expertise in Creative Design. "
            "Core skills include: Adobe Photoshop, Illustrator, Brand Identity. "
            "Work history: Graphic Designer at Tribal DDB Cairo. "
            "Prefers Hybrid work in Cairo, Egypt.",
            "Job Title: Senior Graphic Designer. "
            "Company: Tribal DDB Cairo. Industry: Advertising. "
            "Required Skills: Photoshop, Illustrator, Brand Design, Typography. "
            "Work Mode: Hybrid. Location: Cairo, Egypt. "
            "Description: Create compelling visual campaigns for brand accounts.",
        ], label=0.89),

    # ---- Positive Pairs: Digital Marketing ----
    InputExample(
        texts=[
            "Professional candidate with expertise in Digital Marketing. "
            "Core skills include: SEO, Google Ads, Social Media Marketing, Analytics. "
            "Work history: Digital Marketing Specialist at Jumia Egypt. "
            "Prefers Remote work in Cairo, Egypt.",
            "Job Title: Digital Marketing Manager. "
            "Company: Jumia Egypt. Industry: E-Commerce. "
            "Required Skills: SEO, SEM, Google Analytics, Social Media. "
            "Work Mode: Remote. Location: Cairo, Egypt. "
            "Description: Drive user acquisition for Africa's largest e-commerce marketplace.",
        ], label=0.90),

    # ---- Positive Pairs: DevOps ----
    InputExample(
        texts=[
            "Professional candidate with expertise in DevOps Engineering. "
            "Core skills include: Docker, Kubernetes, Terraform, AWS, CI/CD. "
            "Work history: DevOps Engineer at Si-Ware Systems. "
            "Prefers Remote work in Cairo, Egypt.",
            "Job Title: Senior DevOps Engineer. "
            "Company: Si-Ware Systems. Industry: Semiconductor Technology. "
            "Required Skills: Docker, Kubernetes, Terraform, AWS, Jenkins. "
            "Work Mode: Remote. Location: Cairo, Egypt. "
            "Description: Build and maintain cloud infrastructure and CI/CD pipelines.",
        ], label=0.91),

    # ---- Positive Pairs: Data Analysis ----
    InputExample(
        texts=[
            "Professional candidate with expertise in Data Analysis. "
            "Core skills include: SQL, Excel, Power BI, Python, Statistics. "
            "Work history: Business Analyst at Orange Egypt. "
            "Prefers Onsite work in Cairo, Egypt.",
            "Job Title: Data Analyst. "
            "Company: Orange Egypt. Industry: Telecommunications. "
            "Required Skills: SQL, Power BI, Excel, Python, Data Visualization. "
            "Work Mode: Onsite. Location: Cairo, Egypt. "
            "Description: Transform raw telecom data into actionable business insights.",
        ], label=0.89),

    # ---- Positive Pairs: Cross-domain ----
    InputExample(
        texts=[
            "Professional candidate with expertise in Software Engineering. "
            "Core skills include: Python, FastAPI, Docker, PostgreSQL. "
            "Work history: Backend Developer at MaxAB. "
            "Prefers Remote work in Cairo, Egypt.",
            "Job Title: Python Backend Developer. "
            "Company: Capiter. Industry: B2B E-Commerce. "
            "Required Skills: Python, FastAPI, Docker, PostgreSQL, Redis. "
            "Work Mode: Remote. Location: Cairo, Egypt. "
            "Description: Build supply chain management APIs for B2B marketplace.",
        ], label=0.87),
    InputExample(
        texts=[
            "Professional candidate with expertise in Mobile Development. "
            "Core skills include: Flutter, Dart, Firebase, Provider, Bloc. "
            "Work history: Junior Mobile Developer at Robusta Studio. "
            "Prefers Onsite work in Alexandria, Egypt.",
            "Job Title: Mobile Application Developer. "
            "Company: Robusta Studio. Industry: Software Consultancy. "
            "Required Skills: Flutter, Dart, Firebase, State Management. "
            "Work Mode: Onsite. Location: Alexandria, Egypt. "
            "Description: Develop cross-platform mobile apps for diverse client projects.",
        ], label=0.92),

    # ---- Negative Pairs: Mismatches ----
    InputExample(
        texts=[
            "Professional candidate with expertise in Graphic Design. "
            "Core skills include: Adobe Photoshop, Illustrator, Brand Identity. "
            "Work history: Visual Designer at Leo Burnett Cairo. "
            "Prefers Onsite work in Cairo, Egypt.",
            "Job Title: Senior DevOps Engineer. "
            "Company: Amazon Web Services. Industry: Cloud Computing. "
            "Required Skills: Kubernetes, Terraform, AWS, Linux, CI/CD. "
            "Work Mode: Remote. Location: Cairo, Egypt. "
            "Description: Design and operate large-scale cloud infrastructure.",
        ], label=0.12),
    InputExample(
        texts=[
            "Professional candidate with expertise in Video Production. "
            "Core skills include: Adobe Premiere Pro, After Effects, Color Grading. "
            "Work history: Video Editor at ON TV. "
            "Prefers Onsite work in Cairo, Egypt.",
            "Job Title: Senior Backend Python Developer. "
            "Company: Microsoft Egypt. Industry: Technology. "
            "Required Skills: Python, Django, PostgreSQL, Microservices. "
            "Work Mode: Hybrid. Location: Cairo, Egypt. "
            "Description: Build enterprise-grade backend services at scale.",
        ], label=0.10),
    InputExample(
        texts=[
            "Professional candidate with expertise in Digital Marketing. "
            "Core skills include: Social Media Marketing, Content Strategy, SEO. "
            "Work history: Marketing Manager at Careem Egypt. "
            "Prefers Remote work in Cairo, Egypt.",
            "Job Title: Machine Learning Engineer. "
            "Company: Valeo Egypt. Industry: Automotive Technology. "
            "Required Skills: Python, PyTorch, TensorFlow, Computer Vision. "
            "Work Mode: Hybrid. Location: Cairo, Egypt. "
            "Description: Build deep learning models for ADAS systems.",
        ], label=0.15),
    InputExample(
        texts=[
            "Professional candidate with expertise in Human Resources. "
            "Core skills include: Recruitment, Employee Relations, HR Policies. "
            "Work history: HR Specialist at Teleperformance Egypt. "
            "Prefers Onsite work in Cairo, Egypt.",
            "Job Title: Flutter Mobile Developer. "
            "Company: Swvl. Industry: Transportation Technology. "
            "Required Skills: Flutter, Dart, Firebase, REST APIs. "
            "Work Mode: Remote. Location: Cairo, Egypt. "
            "Description: Build cross-platform mobile transportation apps.",
        ], label=0.10),
]

n_positive = sum(1 for ex in train_examples if ex.label >= 0.5)
n_negative = len(train_examples) - n_positive
print(f"\u2705 Training dataset ready: {len(train_examples)} pairs ({n_positive} positive, {n_negative} negative)")

## Step 3: Configure Training Pipeline

We use **CosineSimilarityLoss** which:
1. Encodes both texts in each pair through the model
2. Computes cosine similarity between the two embeddings
3. Compares the computed similarity with the target label
4. Backpropagates the error to update the model weights

This teaches the model to produce **similar vectors** for matching profile-job pairs and **dissimilar vectors** for non-matching pairs.

In [ ]:
# Training hyperparameters
EPOCHS = 4
BATCH_SIZE = 4
WARMUP_RATIO = 0.1

# Load a fresh copy of the base model for training
model = SentenceTransformer(BASE_MODEL_NAME)

# Configure data loader and loss function
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss = losses.CosineSimilarityLoss(model)

total_steps = len(train_dataloader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

print(f"\u2699\ufe0f Training Configuration:")
print(f"   Loss function:    CosineSimilarityLoss")
print(f"   Batch size:       {BATCH_SIZE}")
print(f"   Epochs:           {EPOCHS}")
print(f"   Total steps:      {total_steps}")
print(f"   Warmup steps:     {warmup_steps}")

## Step 4: Fine-Tune the Model

This cell runs the actual training loop. The model weights are updated via backpropagation through the transformer layers.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("../local_storage/custom_kindred_model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\U0001f3cb\ufe0f Starting fine-tuning ({EPOCHS} epochs)...")
start_train = time.time()

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    output_path=str(OUTPUT_DIR),
)

train_time = time.time() - start_train
print(f"\n\u2705 Fine-tuning complete in {train_time:.1f}s")

## Step 5: Save the Fine-Tuned Model

In [ ]:
model.save(str(OUTPUT_DIR))

abs_path = OUTPUT_DIR.resolve()
saved_files = list(OUTPUT_DIR.rglob("*"))
saved_size_mb = sum(f.stat().st_size for f in saved_files if f.is_file()) / (1024 * 1024)

print(f"\U0001f4be Model saved successfully!")
print(f"   Path: {abs_path}")
print(f"   Files: {sum(1 for f in saved_files if f.is_file())}")
print(f"   Size: {saved_size_mb:.1f} MB")

## Step 6: Verify Output Dimensions

Critical check: The fine-tuned model must still produce **384-dimensional** embeddings to remain compatible with our ChromaDB vector database.

In [ ]:
EXPECTED_DIMENSIONS = 384

# Reload from disk to verify the save was clean
verified_model = SentenceTransformer(str(OUTPUT_DIR))

test_sentence = (
    "Professional candidate with expertise in Software Engineering. "
    "Core skills include: Flutter, Dart, Firebase. "
    "Prefers Remote work in Cairo, Egypt."
)

test_embedding = verified_model.encode(test_sentence, normalize_embeddings=True)
actual_dim = len(test_embedding)

print(f"Test embedding shape: {test_embedding.shape}")
print(f"Dimensions: {actual_dim}")
print(f"Normalized (L2 norm): {np.linalg.norm(test_embedding):.4f}")
print()

if actual_dim == EXPECTED_DIMENSIONS:
    print(f"\u2705 DIMENSION CHECK PASSED: {actual_dim} dimensions")
    print(f"   Compatible with existing ChromaDB schema.")
else:
    print(f"\u274c DIMENSION CHECK FAILED: expected {EXPECTED_DIMENSIONS}, got {actual_dim}")

## Step 7: Before vs. After Comparison

This is the most important validation: we compare cosine similarity scores between the **base model** and the **fine-tuned model** on profile-job pairs to demonstrate that fine-tuning improved domain-specific matching.

In [ ]:
from sentence_transformers import util

# Test pairs: (profile_text, job_text, description)
test_pairs = [
    (
        "Professional candidate with expertise in Mobile Development. "
        "Core skills include: Flutter, Dart, Firebase, Provider. "
        "Work history: Mobile Developer at Swvl. "
        "Prefers Remote work in Cairo, Egypt.",
        
        "Job Title: Flutter Developer. "
        "Company: Careem. Industry: Transportation Technology. "
        "Required Skills: Flutter, Dart, Firebase, REST APIs. "
        "Work Mode: Remote. Location: Cairo, Egypt. "
        "Description: Build cross-platform ride-hailing mobile apps.",
        
        "Flutter Dev \u2194 Flutter Job (should be HIGH)"
    ),
    (
        "Professional candidate with expertise in Data Science. "
        "Core skills include: Python, TensorFlow, pandas, SQL. "
        "Work history: Data Scientist at Fawry. "
        "Prefers Hybrid work in Cairo, Egypt.",
        
        "Job Title: ML Engineer. "
        "Company: Instabug. Industry: Technology. "
        "Required Skills: Python, PyTorch, scikit-learn, MLOps. "
        "Work Mode: Hybrid. Location: Cairo, Egypt. "
        "Description: Build ML pipelines for crash analytics.",
        
        "Data Scientist \u2194 ML Engineer (should be MEDIUM-HIGH)"
    ),
    (
        "Professional candidate with expertise in Graphic Design. "
        "Core skills include: Photoshop, Illustrator, Brand Design. "
        "Work history: Graphic Designer at Leo Burnett Cairo. "
        "Prefers Onsite work in Cairo, Egypt.",
        
        "Job Title: Senior DevOps Engineer. "
        "Company: AWS. Industry: Cloud Computing. "
        "Required Skills: Kubernetes, Terraform, Docker, Linux. "
        "Work Mode: Remote. Location: Cairo, Egypt. "
        "Description: Manage large-scale cloud infrastructure.",
        
        "Graphic Designer \u2194 DevOps Engineer (should be LOW)"
    ),
]

print("\U0001f4ca Before vs. After Fine-Tuning Comparison")
print("=" * 65)
print(f"{'Test Case':<45} {'Base':>8} {'Tuned':>8} {'Delta':>8}")
print("-" * 65)

for profile, job, desc in test_pairs:
    # Base model similarity
    base_emb_p = base_model.encode(profile, normalize_embeddings=True)
    base_emb_j = base_model.encode(job, normalize_embeddings=True)
    base_sim = float(util.cos_sim(base_emb_p, base_emb_j)[0][0])
    
    # Fine-tuned model similarity
    tuned_emb_p = verified_model.encode(profile, normalize_embeddings=True)
    tuned_emb_j = verified_model.encode(job, normalize_embeddings=True)
    tuned_sim = float(util.cos_sim(tuned_emb_p, tuned_emb_j)[0][0])
    
    delta = tuned_sim - base_sim
    arrow = "\u2b06\ufe0f" if delta > 0.005 else ("\u2b07\ufe0f" if delta < -0.005 else "\u2796")
    
    print(f"{desc:<45} {base_sim:>7.4f} {tuned_sim:>8.4f} {arrow}{delta:>+7.4f}")

print("-" * 65)
print("\n\u2705 Positive pairs should show INCREASED similarity (\u2b06\ufe0f)")
print("\u2705 Negative pairs should show DECREASED similarity (\u2b07\ufe0f)")

## Activation Instructions

To use the fine-tuned model in the live application:

### 1. Migrate existing ChromaDB jobs (no data loss)
```bash
cd backend
python scripts/migrate_embeddings.py
```

### 2. Update `backend/.env`
```
EMBED_MODEL=./local_storage/custom_kindred_model
```

### 3. Restart the backend
```bash
python main.py
```

The scraper will automatically use the fine-tuned model for all new jobs.  
To **roll back**: revert `EMBED_MODEL` in `.env` and re-run the migration script.